# Evaluate your AI Agent with ADK

## Overview

This notebook shows how to evaluate an **ADK (Agent Development Kit)** agent using `AgentEvaluator.evaluate()`.

We use the weather agent from [building_agent_with_adk.ipynb](../../building_agents/solutions/building_agent_with_adk.ipynb) across all examples. The agent can:

1. **Look up city weather (`get_weather`)**: Return weather reports for `New York`, `London`, and `Tokyo` in `Celsius` or `Fahrenheit`, and return an error message when data is unavailable (such as `Paris`).
2. **Save user preferences (`set_temperature_preference`)**: Store the user's preferred temperature unit (`Celsius` or `Fahrenheit`) in session state (`user:temperature_unit`).
3. **Route to sub-agents (`greeting_agent` and `farewell_agent`)**: Send greetings (`say_hello`) and goodbyes (`say_goodbye`) to sub-agents.

## Learning objectives

In this notebook, you will learn how to:
* Set up an ADK agent with tools, session state, and sub-agents
* Create ADK evaluation datasets (`.test.json`) and config files (`test_config.json`)
* Evaluate single tool usage and check latency and tool call counts
* Evaluate multi-step tool trajectories (`EXACT`, `IN_ORDER`, and `ANY_ORDER`)
* Evaluate responses with custom rubrics and custom Python functions
* Evaluate multi-turn conversations, session state, and sub-agents

For more details, see the [ADK Evaluation documentation](https://google.github.io/adk-docs/evaluate/) and [ADK Evaluation Criteria](https://google.github.io/adk-docs/evaluate/criteria/).

## Import libraries

In [ ]:
import asyncio
import importlib
import json
import logging
import os
import time
import uuid
import warnings

import pandas as pd
from google.adk.agents import Agent
from google.adk.evaluation.agent_evaluator import AgentEvaluator
from google.adk.evaluation.eval_case import get_all_tool_calls
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.tool_context import ToolContext
from google.genai import types
from IPython.display import HTML, Markdown, display

# Ignore all warnings
warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.ERROR)
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)

In [ ]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0].strip()
GEMINI_LOCATION = "global"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = GEMINI_LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

In [ ]:
!mkdir -p adk_agents/agent1_weather_lookup

In [ ]:
%%bash
echo > adk_agents/.env "GOOGLE_CLOUD_PROJECT=$GOOGLE_CLOUD_PROJECT
GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION
GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI
"

## Define helper functions

`await AgentEvaluator.evaluate(...)` takes the following arguments:
* `agent_module`: Python module path containing `root_agent` (for example, `"adk_agents.agent1_weather_lookup.agent"`)
* `eval_dataset_file_path_or_dir`: Path to a `.test.json` file or a directory of `.test.json` files
* `num_runs`: Number of times to run each test case
* `agent_name`: Optional sub-agent name to test by itself
* `initial_session_file`: Optional file with initial session state
* `print_detailed_results`: Whether to print details when a metric falls below its threshold

### Capture evaluation results into a DataFrame

First, we wrap `AgentEvaluator._get_eval_results_by_eval_id` so each call to `AgentEvaluator.evaluate()` saves the per-prompt scores into a pandas DataFrame.

In [ ]:
OBSERVABILITY_METRICS = {
    "tool_call_count",
    "avg_invocation_latency_sec",
}

_LAST_EVAL_RESULTS_DF = pd.DataFrame()
_ORIG_GET_EVAL_RESULTS = AgentEvaluator._get_eval_results_by_eval_id


async def _capturing_get_eval_results_by_eval_id(*args, **kwargs):
    """Wraps AgentEvaluator._get_eval_results_by_eval_id to store results in a DataFrame."""
    global _LAST_EVAL_RESULTS_DF
    start_time = time.perf_counter()
    eval_results_by_eval_id = await _ORIG_GET_EVAL_RESULTS(*args, **kwargs)
    elapsed_sec = time.perf_counter() - start_time

    total_invocations = sum(
        len(cr.eval_metric_result_per_invocation)
        for case_results in eval_results_by_eval_id.values()
        for cr in case_results
    )
    avg_latency = elapsed_sec / max(total_invocations, 1)

    rows = []
    for eval_id, case_results in eval_results_by_eval_id.items():
        for cr in case_results:
            for inv_res in cr.eval_metric_result_per_invocation:
                exp_inv = inv_res.expected_invocation
                act_inv = inv_res.actual_invocation
                base_row = {
                    "eval_set_id": cr.eval_set_id,
                    "eval_id": eval_id,
                    "prompt": AgentEvaluator._convert_content_to_text(
                        exp_inv.user_content if exp_inv else act_inv.user_content
                    ),
                    "expected_response": AgentEvaluator._convert_content_to_text(
                        exp_inv.final_response if exp_inv else None
                    ),
                    "actual_response": AgentEvaluator._convert_content_to_text(
                        act_inv.final_response
                    ),
                    "expected_tool_calls": AgentEvaluator._convert_tool_calls_to_text(
                        exp_inv.intermediate_data if exp_inv else None
                    ),
                    "actual_tool_calls": AgentEvaluator._convert_tool_calls_to_text(
                        act_inv.intermediate_data
                    ),
                }

                for mr in inv_res.eval_metric_results:
                    rows.append({
                        **base_row,
                        "metric_name": mr.metric_name,
                        "threshold": mr.threshold,
                        "score": mr.score,
                        "eval_status": getattr(mr.eval_status, "name", str(mr.eval_status)),
                    })

                num_tools = len(get_all_tool_calls(act_inv.intermediate_data))
                for obs_name, obs_val in [
                    ("tool_call_count", float(num_tools)),
                    ("avg_invocation_latency_sec", round(avg_latency, 3)),
                ]:
                    rows.append({
                        **base_row,
                        "metric_name": obs_name,
                        "threshold": None,
                        "score": obs_val,
                        "eval_status": "OBSERVABILITY",
                    })

    _LAST_EVAL_RESULTS_DF = pd.DataFrame(rows)
    return eval_results_by_eval_id


AgentEvaluator._get_eval_results_by_eval_id = staticmethod(
    _capturing_get_eval_results_by_eval_id
)

Next, we define `run_adk_eval` to call `AgentEvaluator.evaluate()` and return the captured DataFrame.

In [ ]:
def get_last_eval_df() -> pd.DataFrame:
    """Returns the DataFrame from the most recent AgentEvaluator.evaluate() call."""
    return _LAST_EVAL_RESULTS_DF.copy()


async def run_adk_eval(
    eval_dataset_file_path_or_dir: str,
    agent_module: str = "adk_agents.agent1_weather_lookup.agent",
    num_runs: int = 1,
    agent_name: str | None = None,
    initial_session_file: str | None = None,
    print_detailed_results: bool = True,
) -> pd.DataFrame:
    """Runs AgentEvaluator.evaluate() and returns the results DataFrame."""
    try:
        await AgentEvaluator.evaluate(
            agent_module=agent_module,
            eval_dataset_file_path_or_dir=eval_dataset_file_path_or_dir,
            num_runs=num_runs,
            agent_name=agent_name,
            initial_session_file=initial_session_file,
            print_detailed_results=print_detailed_results,
        )
        print("All evaluation criteria passed!")
    except AssertionError as e:
        print(f"Some metrics fell below threshold:\n{e}")

    return get_last_eval_df()

### Display summary tables and per-prompt results

We define `display_eval_report` to show summary tables for quality metrics, latency, and per-prompt scores.

In [ ]:
def display_eval_report(eval_df: pd.DataFrame) -> None:
    """Displays summary tables and per-prompt scores."""
    if eval_df.empty:
        print("No evaluation results to display.")
        return

    is_obs = eval_df["metric_name"].isin(OBSERVABILITY_METRICS) | (
        eval_df["eval_status"].astype(str).str.contains("OBSERVABILITY")
    )
    quality_df = eval_df[~is_obs].copy()
    obs_df = eval_df[is_obs].copy()

    if not quality_df.empty:
        quality_df["passed"] = quality_df["eval_status"].astype(str).str.contains("PASSED")
        quality_summary = (
            quality_df.groupby("metric_name", sort=False)
            .agg(
                mean_score=("score", "mean"),
                threshold=("threshold", "first"),
                pass_rate=("passed", "mean"),
                invocations=("score", "count"),
            )
            .reset_index()
        )
        display(Markdown("### Quality metrics"))
        display(quality_summary)

    if not obs_df.empty:
        obs_summary = (
            obs_df.groupby("metric_name", sort=False)
            .agg(
                mean=("score", "mean"),
                min=("score", "min"),
                max=("score", "max"),
                total=("score", "sum"),
            )
            .reset_index()
        )
        display(Markdown("### Latency and tool calls"))
        display(obs_summary)

    pivot_scores = eval_df.pivot_table(
        index=["eval_id", "prompt"],
        columns="metric_name",
        values="score",
        aggfunc="mean",
        sort=False,
    ).reset_index()
    display(Markdown("### Results by prompt"))
    display(pivot_scores)

We also define `render_trajectory_drilldown` and `display_dataframe_rows` to inspect individual test cases and compare expected vs. actual tool calls side by side.

In [ ]:
def render_trajectory_drilldown(expected_calls: str, actual_calls: str) -> None:
    """Renders expected vs. actual tool calls side by side."""
    exp_str = expected_calls if pd.notna(expected_calls) and str(expected_calls).strip() else "(none)"
    act_str = actual_calls if pd.notna(actual_calls) and str(actual_calls).strip() else "(none)"

    drilldown_html = f"""
    <div style="display: flex; gap: 20px; font-family: monospace; font-size: 13px; background: #f8f9fa; padding: 10px; border-radius: 6px; margin-top: 6px; margin-bottom: 12px;">
        <div style="flex: 1;">
            <strong>Expected tool calls:</strong>
            <pre style="margin: 6px 0 0 0; white-space: pre-wrap;">{exp_str}</pre>
        </div>
        <div style="flex: 1;">
            <strong>Actual tool calls:</strong>
            <pre style="margin: 6px 0 0 0; white-space: pre-wrap;">{act_str}</pre>
        </div>
    </div>
    """
    display(HTML(drilldown_html))


def display_dataframe_rows(
    df: pd.DataFrame,
    columns: list[str] | None = None,
    num_rows: int = 4,
    display_drilldown: bool = False,
) -> None:
    """Displays rows from a DataFrame and optionally shows tool call comparisons."""
    base_style = "font-family: monospace; font-size: 13px; white-space: pre-wrap; width: auto; overflow-x: auto;"
    header_style = base_style + "font-weight: bold;"

    if {"eval_id", "metric_name", "prompt"}.issubset(df.columns):
        grouped = list(df.groupby(["eval_id", "prompt"], sort=False))
        for (eval_id, prompt), group in grouped[:num_rows]:
            first_row = group.iloc[0]
            display(HTML(f"<span style='{header_style}'>Eval ID: </span><span style='{base_style}'>{eval_id}</span><br>"))
            display(HTML(f"<span style='{header_style}'>Prompt: </span><span style='{base_style}'>{prompt}</span><br>"))
            display(HTML(f"<span style='{header_style}'>Expected Response: </span><span style='{base_style}'>{first_row.get('expected_response', '')}</span><br>"))
            display(HTML(f"<span style='{header_style}'>Actual Response: </span><span style='{base_style}'>{first_row.get('actual_response', '')}</span><br>"))

            metrics_str = ", ".join(
                f"{r['metric_name']}={r['score']:.2f}" if pd.notna(r["score"]) else f"{r['metric_name']}=N/A"
                for _, r in group.iterrows()
            )
            display(HTML(f"<span style='{header_style}'>Scores: </span><span style='{base_style}'>{metrics_str}</span><br>"))

            if display_drilldown:
                render_trajectory_drilldown(
                    first_row.get("expected_tool_calls", ""),
                    first_row.get("actual_tool_calls", ""),
                )
            display(HTML("<hr>"))
        return

    if columns:
        df = df[[c for c in columns if c in df.columns]]
    for _, row in df.head(num_rows).iterrows():
        for column in df.columns:
            display(HTML(f"<span style='{header_style}'>{column.replace('_', ' ').title()}: </span>"))
            display(HTML(f"<span style='{base_style}'>{row[column]}</span><br>"))
        display(HTML("<hr>"))

### Plot metric scores

Finally, we define `plot_bar_plot` and `display_radar_plot` to plot metric scores across evaluation runs.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"


def _extract_quality_summary(
    eval_df: pd.DataFrame, metrics: list[str] | None = None
) -> dict[str, float]:
    """Extracts mean score per metric from an AgentEvaluator DataFrame or dict."""
    if isinstance(eval_df, dict):
        return eval_df
    df = eval_df.copy()
    if metrics:
        df = df[df["metric_name"].isin(metrics)]
    else:
        df = df[~df["metric_name"].isin(OBSERVABILITY_METRICS)]
    return df.groupby("metric_name", sort=False)["score"].mean().dropna().to_dict()


def plot_bar_plot(
    eval_df: pd.DataFrame | dict[str, float],
    title: str,
    metrics: list[str] | None = None,
) -> None:
    """Displays a bar chart of mean metric scores."""
    summary_metrics = _extract_quality_summary(eval_df, metrics=metrics)
    fig = go.Figure(
        data=[
            go.Bar(
                x=list(summary_metrics.keys()),
                y=list(summary_metrics.values()),
                text=[f"{v:.2f}" for v in summary_metrics.values()],
                textposition="auto",
                name=title,
            )
        ]
    )
    fig.update_layout(title=title, barmode="group", yaxis=dict(range=[0, 1.08]))
    fig.show()


def display_radar_plot(
    eval_df: pd.DataFrame | dict[str, float],
    title: str,
    metrics: list[str] | None = None,
) -> None:
    """Plots a radar chart of mean metric scores."""
    summary_metrics = _extract_quality_summary(eval_df, metrics=metrics)
    fig = go.Figure()
    max_val = max(list(summary_metrics.values()) + [1.0])
    fig.add_trace(
        go.Scatterpolar(
            r=list(summary_metrics.values()),
            theta=list(summary_metrics.keys()),
            fill="toself",
            name=title,
        )
    )
    fig.update_layout(
        title=title,
        polar=dict(radialaxis=dict(visible=True, range=[0, max_val])),
        showlegend=True,
    )
    fig.show()

## Build the weather agent

We will define the weather agent from [building_agent_with_adk.ipynb](./building_agent_with_adk.ipynb) inside `./adk_agents/agent1_weather_lookup`:

1. **`get_weather(city, tool_context)`**: Looks up weather for `New York`, `London`, or `Tokyo`, formats the temperature in `Celsius` or `Fahrenheit` based on `user:temperature_unit` in session state, and returns an error dictionary for unsupported cities like `Paris`.
2. **`set_temperature_preference(unit, tool_context)`**: Updates `user:temperature_unit` (`"Celsius"` or `"Fahrenheit"`) in session state.
3. **`greeting_agent` and `farewell_agent` sub-agents**: Handle greetings (`say_hello`) and farewells (`say_goodbye`).

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/tools.py
from google.adk.tools.tool_context import ToolContext


def get_weather(city: str, tool_context: ToolContext = None) -> dict:
    """Retrieves the current weather report for a specified city.

    Args:
        city (str): The name of the city (e.g., "New York", "London", "Tokyo").

    Returns:
        dict: A dictionary containing the weather information.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'report' key with weather details.
              If 'error', includes an 'error_message' key.
    """
    print(f"--- Tool: get_weather called for city: {city} ---")
    city_normalized = city.lower().replace(" ", "")

    preferred_unit = "Celsius"
    if tool_context is not None:
        preferred_unit = tool_context.state.get("user:temperature_unit", "Celsius")
        tool_context.state["last_city_checked"] = city

    mock_weather_db = {
        "newyork": {"temp_c": 25, "condition": "sunny", "display_name": "New York"},
        "london": {"temp_c": 15, "condition": "cloudy", "display_name": "London"},
        "tokyo": {"temp_c": 18, "condition": "experiencing light rain", "display_name": "Tokyo"},
    }

    if city_normalized in mock_weather_db:
        entry = mock_weather_db[city_normalized]
        temp_c = entry["temp_c"]
        if preferred_unit.lower() == "fahrenheit":
            temp_val = round((temp_c * 9 / 5) + 32)
            temp_str = f"{temp_val}°F"
        else:
            temp_str = f"{temp_c}°C"

        if entry["display_name"] == "Tokyo":
            report = f"Tokyo is {entry['condition']} and a temperature of {temp_str}."
        elif entry["display_name"] == "London":
            report = f"It's {entry['condition']} in London with a temperature of {temp_str}."
        else:
            report = f"The weather in {entry['display_name']} is {entry['condition']} with a temperature of {temp_str}."

        return {"status": "success", "report": report}
    else:
        return {
            "status": "error",
            "error_message": f"Sorry, I don't have weather information for '{city}'.",
        }


def set_temperature_preference(unit: str, tool_context: ToolContext = None) -> dict:
    """Sets the user's preferred temperature unit in session state.

    Args:
        unit (str): The preferred temperature unit ("Celsius" or "Fahrenheit").

    Returns:
        dict: Confirmation of the updated preference.
    """
    normalized_unit = "Fahrenheit" if unit.strip().lower().startswith("f") else "Celsius"
    print(f"--- Tool: set_temperature_preference called with unit: {normalized_unit} ---")
    if tool_context is not None:
        tool_context.state["user:temperature_unit"] = normalized_unit
    return {
        "status": "success",
        "message": f"Preferred temperature unit set to {normalized_unit}.",
    }


def say_hello(name: str = "there") -> str:
    """Provides a friendly greeting, optionally addressing the user by name.

    Args:
        name (str): The name of the person to greet. Defaults to "there".

    Returns:
        str: A friendly greeting message.
    """
    if not name or not name.strip():
        name = "there"
    print(f"--- Tool: say_hello called with name: {name} ---")
    return f"Hello, {name}!"


def say_goodbye() -> str:
    """Provides a polite farewell message to conclude the conversation."""
    print("--- Tool: say_goodbye called ---")
    return "Goodbye! Have a great day."

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/agent.py
from google.adk.agents import Agent

from .tools import (
    get_weather,
    say_goodbye,
    say_hello,
    set_temperature_preference,
)

MODEL = "gemini-3.8-flash"

greeting_agent = Agent(
    name="greeting_agent",
    model=MODEL,
    description="Handles simple greetings and hellos using the 'say_hello' tool.",
    instruction=(
        "You are the Greeting Agent. Your task is to provide a friendly greeting "
        "using the 'say_hello' tool. If the user provides their name, pass it to the tool. "
        "If the user asks about weather or temperature preferences, transfer to 'weather_agent_v1'."
    ),
    tools=[say_hello],
)

farewell_agent = Agent(
    name="farewell_agent",
    model=MODEL,
    description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
    instruction=(
        "You are the Farewell Agent. Your task is to provide a polite goodbye message "
        "using the 'say_goodbye' tool when the user ends the conversation. "
        "If the user asks about weather or temperature preferences, transfer to 'weather_agent_v1'."
    ),
    tools=[say_goodbye],
)

root_agent = Agent(
    name="weather_agent_v1",
    model=MODEL,
    description="Provides city weather reports, manages temperature unit preferences, and delegates greetings/farewells.",
    instruction=(
        "You are a helpful Weather Assistant.\n"
        "- When the user asks for the weather in one or more cities, call 'get_weather' for each city "
        "in the exact order the cities are mentioned by the user.\n"
        "- If the user asks to change or set their preferred temperature unit (Celsius or Fahrenheit), "
        "call 'set_temperature_preference' BEFORE calling 'get_weather'.\n"
        "- If 'get_weather' returns an error for a city, politely inform the user that weather data "
        "is unavailable for that city and never invent or guess weather conditions.\n"
        "- If 'get_weather' succeeds, clearly state the weather report returned by the tool.\n"
        "- Delegate standalone greetings to 'greeting_agent' and standalone farewells to 'farewell_agent'."
    ),
    tools=[get_weather, set_temperature_preference],
    sub_agents=[greeting_agent, farewell_agent],
)

In [ ]:
with open("./adk_agents/agent1_weather_lookup/__init__.py", "w") as f:
    f.write("from . import agent\n")

import sys

if "." not in sys.path:
    sys.path.insert(0, ".")

import adk_agents.agent1_weather_lookup.agent as weather_agent_module

importlib.reload(weather_agent_module)
root_agent = weather_agent_module.root_agent
print(f"Loaded root_agent: {root_agent.name} with model {root_agent.model}")

### Test the agent

Before running `AgentEvaluator.evaluate()`, let's send one test query using `Runner` and `InMemorySessionService` to see the tool calls and session state.

In [ ]:
APP_NAME = "weather_info_app"
USER_ID = "user_1"
SESSION_ID = "session_001"


async def call_agent_async(query: str, session_id: str | None = None) -> dict:
    """Runs a single query against the weather agent."""
    session_service = InMemorySessionService()
    resolved_session_id = session_id or f"session_{uuid.uuid4().hex[:8]}"
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=resolved_session_id
    )
    runner = Runner(
        agent=agent.root_agent, app_name=APP_NAME, session_service=session_service
    )

    content = types.Content(role="user", parts=[types.Part(text=query)])
    events = runner.run_async(
        user_id=USER_ID, session_id=resolved_session_id, new_message=content
    )

    final_response = ""
    tool_calls = []
    async for event in events:
        if not event.content or not event.content.parts:
            continue
        for part in event.content.parts:
            if part.function_call:
                tool_calls.append(
                    {"name": part.function_call.name, "args": dict(part.function_call.args)}
                )
        if event.is_final_response() and event.content.parts[0].text:
            final_response = event.content.parts[0].text.strip()

    updated_session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=resolved_session_id
    )
    return {
        "query": query,
        "tool_calls": tool_calls,
        "response": final_response,
        "session_state": dict(updated_session.state) if updated_session else {},
    }


sample_outcome = await call_agent_async(
    "Set my preferred temperature unit to Fahrenheit and tell me the weather in Tokyo."
)
print("Tool Calls:", sample_outcome["tool_calls"])
print("Final Response:", sample_outcome["response"])
print("Session State:", sample_outcome["session_state"])

## Evaluate the agent

`AgentEvaluator.evaluate()` uses two JSON files:

1. **Test file (`.test.json`)**: Contains `eval_cases`. Each test case includes:
   * `eval_id`: Name of the test case
   * `conversation`: List of turns with `user_content` (prompt), `final_response` (expected answer), and `intermediate_data.tool_uses` (expected tool calls and arguments)
   * `session_input`: Initial `app_name`, `user_id`, and `state`
2. **Config file (`test_config.json`)**: Placed in the same folder as the `.test.json` file. It defines `criteria` (metrics and thresholds) and optional `custom_metrics`.

### Prepare the evaluation dataset

Let's create `weather_assistant_core.test.json` with 6 test cases:
1. **`new_york_weather`**: Weather in `New York` (`25°C`, sunny)
2. **`london_weather`**: Weather in `London` (`15°C`, cloudy)
3. **`paris_unsupported_city`**: Unsupported city (`Paris`)
4. **`multi_city_ny_london`**: Two cities in order (`New York`, then `London`)
5. **`preference_then_tokyo_fahrenheit`**: Change temperature unit to `Fahrenheit`, then check `Tokyo` (`64°F`)
6. **`three_cities_mixed_support`**: Three cities (`New York`, `London`, and `Paris`)

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/weather_assistant_core.test.json
{
  "eval_set_id": "weather_assistant_core_eval_set",
  "name": "Weather Agent Evaluation Suite",
  "description": "Test cases for city weather lookups, unsupported cities, multi-city queries, and temperature unit preferences.",
  "eval_cases": [
    {
      "eval_id": "new_york_weather",
      "conversation": [
        {
          "invocation_id": "inv-ny-001",
          "user_content": {
            "parts": [{"text": "Tell me the weather in New York"}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "The weather in New York is sunny with a temperature of 25°C."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "get_weather", "args": {"city": "New York"}}
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "weather_info_app",
        "user_id": "user_1",
        "state": {}
      }
    },
    {
      "eval_id": "london_weather",
      "conversation": [
        {
          "invocation_id": "inv-lon-001",
          "user_content": {
            "parts": [{"text": "What is the weather like in London?"}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "It's cloudy in London with a temperature of 15°C."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "get_weather", "args": {"city": "London"}}
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "weather_info_app",
        "user_id": "user_1",
        "state": {}
      }
    },
    {
      "eval_id": "paris_unsupported_city",
      "conversation": [
        {
          "invocation_id": "inv-par-001",
          "user_content": {
            "parts": [{"text": "How about Paris?"}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "Sorry, I don't have weather information for Paris."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "get_weather", "args": {"city": "Paris"}}
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "weather_info_app",
        "user_id": "user_1",
        "state": {}
      }
    },
    {
      "eval_id": "multi_city_ny_london",
      "conversation": [
        {
          "invocation_id": "inv-nylon-001",
          "user_content": {
            "parts": [{"text": "Tell me the weather in New York and London?"}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "The weather in New York is sunny with a temperature of 25°C, and in London it is cloudy with a temperature of 15°C."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "get_weather", "args": {"city": "New York"}},
              {"name": "get_weather", "args": {"city": "London"}}
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "weather_info_app",
        "user_id": "user_1",
        "state": {}
      }
    },
    {
      "eval_id": "preference_then_tokyo_fahrenheit",
      "conversation": [
        {
          "invocation_id": "inv-tok-001",
          "user_content": {
            "parts": [{"text": "Set my preferred temperature unit to Fahrenheit and tell me the weather in Tokyo."}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "Your preferred temperature unit has been set to Fahrenheit. Tokyo is experiencing light rain with a temperature of 64°F."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "set_temperature_preference", "args": {"unit": "Fahrenheit"}},
              {"name": "get_weather", "args": {"city": "Tokyo"}}
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "weather_info_app",
        "user_id": "user_1",
        "state": {}
      }
    },
    {
      "eval_id": "three_cities_mixed_support",
      "conversation": [
        {
          "invocation_id": "inv-3city-001",
          "user_content": {
            "parts": [{"text": "Tell me the weather in New York, London, and Paris?"}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "The weather in New York is sunny with a temperature of 25°C, and London is cloudy with a temperature of 15°C. However, I don't have weather information for Paris."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "get_weather", "args": {"city": "New York"}},
              {"name": "get_weather", "args": {"city": "London"}},
              {"name": "get_weather", "args": {"city": "Paris"}}
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "weather_info_app",
        "user_id": "user_1",
        "state": {}
      }
    }
  ]
}

Preview the test cases in `weather_assistant_core.test.json`:

In [ ]:
with open("./adk_agents/agent1_weather_lookup/weather_assistant_core.test.json") as f:
    core_eval_set_dict = json.load(f)

eval_dataset_preview = pd.DataFrame(
    [
        {
            "eval_id": case["eval_id"],
            "prompt": case["conversation"][0]["user_content"]["parts"][0]["text"],
            "expected_tool_uses": json.dumps(
                case["conversation"][0]["intermediate_data"]["tool_uses"]
            ),
            "expected_response": case["conversation"][0]["final_response"]["parts"][0]["text"],
        }
        for case in core_eval_set_dict["eval_cases"]
    ]
)
display(eval_dataset_preview)

## Single tool usage evaluation

First, we check if the agent calls `get_weather` with the right city argument on single-city requests.

In `test_config.json`, we set `tool_trajectory_avg_score` with `"match_type": "ANY_ORDER"`, which checks that the expected `get_weather` call occurred during the turn.

In [ ]:
# Create a 3-case file from the single-city cases
single_tool_eval_set = {
    "eval_set_id": "weather_single_tool_eval_set",
    "name": "Weather Agent Single-Tool Suite",
    "eval_cases": core_eval_set_dict["eval_cases"][:3],
}
with open("./adk_agents/agent1_weather_lookup/single_tool_weather.test.json", "w") as f:
    json.dump(single_tool_eval_set, f, indent=2)

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/test_config.json
{
  "criteria": {
    "tool_trajectory_avg_score": {
      "threshold": 1.0,
      "match_type": "ANY_ORDER"
    }
  }
}

Run `AgentEvaluator.evaluate()` on `single_tool_weather.test.json`:

In [ ]:
single_tool_eval_df = await run_adk_eval(
    eval_dataset_file_path_or_dir="./adk_agents/agent1_weather_lookup/single_tool_weather.test.json",
    num_runs=1,
    print_detailed_results=True,
)

display_eval_report(single_tool_eval_df)

In [ ]:
display_dataframe_rows(single_tool_eval_df, num_rows=3, display_drilldown=True)

## Trajectory evaluation

Next, we evaluate multi-step tool sequences across all 6 cases in `weather_assistant_core.test.json`, including multi-city queries (`New York` then `London`) and setting `Fahrenheit` before calling `get_weather("Tokyo")`.

`tool_trajectory_avg_score` supports three `match_type` options in `test_config.json`:

* **`"EXACT"`**: Tool calls and arguments must match the expected list in the same order, with no extra or missing calls.
* **`"IN_ORDER"`**: All expected tool calls must appear in the same order, but extra tool calls in between are allowed.
* **`"ANY_ORDER"`**: All expected tool calls must appear in any order, and extra tool calls are allowed.

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/test_config.json
{
  "criteria": {
    "tool_trajectory_avg_score": {
      "threshold": 1.0,
      "match_type": "EXACT"
    }
  }
}

In [ ]:
trajectory_exact_df = await run_adk_eval(
    eval_dataset_file_path_or_dir="./adk_agents/agent1_weather_lookup/weather_assistant_core.test.json",
    num_runs=1,
)

display_eval_report(trajectory_exact_df)

View the expected vs. actual tool calls for each prompt, and compare `"EXACT"`, `"IN_ORDER"`, and `"ANY_ORDER"` on a bar chart.

In [ ]:
display_dataframe_rows(trajectory_exact_df, num_rows=6, display_drilldown=True)

In [ ]:
trajectory_comparison_scores = {}

for match_type in ["EXACT", "IN_ORDER", "ANY_ORDER"]:
    config = {
        "criteria": {
            "tool_trajectory_avg_score": {
                "threshold": 1.0,
                "match_type": match_type,
            }
        }
    }
    with open("./adk_agents/agent1_weather_lookup/test_config.json", "w") as f:
        json.dump(config, f, indent=2)

    df_mode = await run_adk_eval(
        eval_dataset_file_path_or_dir="./adk_agents/agent1_weather_lookup/weather_assistant_core.test.json",
        num_runs=1,
    )
    score_rows = df_mode[df_mode["metric_name"] == "tool_trajectory_avg_score"]
    trajectory_comparison_scores[f"tool_trajectory ({match_type})"] = score_rows["score"].mean()

plot_bar_plot(
    trajectory_comparison_scores,
    title="Trajectory evaluation by match type",
)

## Custom metrics evaluation

You can also define custom rules in `test_config.json` to check both tool calls and final responses:

### Rubric metrics

`rubric_based_tool_use_quality_v1` and `rubric_based_final_response_quality_v1` use an LLM judge (`gemini-3.8-flash`) to score each turn (`1.0` or `0.0`) against custom rubrics:
* **`rubric_based_tool_use_quality_v1`**:
  * `preference_before_weather`: Call `set_temperature_preference` before `get_weather` when the user changes temperature units.
  * `all_requested_cities_queried`: Call `get_weather` for every city in the prompt.
* **`rubric_based_final_response_quality_v1`**:
  * `accurate_temperature_and_unit`: State the temperature and unit (`°C` or `°F`) returned by `get_weather`.
  * `graceful_unsupported_city_handling`: When `get_weather` returns an error (such as `Paris`), state that weather data is unavailable and do not make up a forecast.

### Custom Python metrics

You can also write a Python function in `custom_eval_metrics.py`, register it in `DEFAULT_METRIC_EVALUATOR_REGISTRY`, and add it under `custom_metrics` in `test_config.json`. Below, `weather_unit_and_error_check` checks that:
1. If `set_temperature_preference(unit="Fahrenheit")` was called, the response contains `"°F"`.
2. If only `Paris` was queried, the response does not contain `"°"`.
3. Otherwise, the response contains `"°C"`.

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/custom_eval_metrics.py
from typing import Optional

from google.adk.evaluation.eval_case import (
    ConversationScenario,
    Invocation,
    get_all_tool_calls,
)
from google.adk.evaluation.eval_metrics import EvalMetric
from google.adk.evaluation.evaluator import (
    EvalStatus,
    EvaluationResult,
    PerInvocationResult,
)


def weather_unit_and_error_check(
    eval_metric: EvalMetric,
    actual_invocations: list[Invocation],
    expected_invocations: Optional[list[Invocation]] = None,
    conversation_scenario: Optional[ConversationScenario] = None,
) -> EvaluationResult:
    """Checks temperature unit formatting (°C vs °F) and Paris error handling."""
    threshold = (
        eval_metric.criterion.threshold
        if eval_metric.criterion and eval_metric.criterion.threshold is not None
        else 1.0
    )
    per_invocation_results = []
    scores = []

    expected_list = expected_invocations or [None] * len(actual_invocations)

    for actual, expected in zip(actual_invocations, expected_list):
        response_text = ""
        if actual.final_response and actual.final_response.parts:
            response_text = "\n".join(
                p.text for p in actual.final_response.parts if p.text
            )

        tool_calls = get_all_tool_calls(actual.intermediate_data)
        called_fahrenheit = any(
            tc.name == "set_temperature_preference"
            and str((tc.args or {}).get("unit", "")).lower().startswith("f")
            for tc in tool_calls
        )
        queried_cities = [
            str((tc.args or {}).get("city", "")).lower()
            for tc in tool_calls
            if tc.name == "get_weather"
        ]

        if called_fahrenheit:
            score = 1.0 if "°F" in response_text else 0.0
        elif queried_cities == ["paris"]:
            score = 1.0 if "°" not in response_text else 0.0
        else:
            score = 1.0 if "°C" in response_text else 0.0

        status = EvalStatus.PASSED if score >= threshold else EvalStatus.FAILED
        per_invocation_results.append(
            PerInvocationResult(
                actual_invocation=actual,
                expected_invocation=expected,
                score=score,
                eval_status=status,
            )
        )
        scores.append(score)

    overall_score = sum(scores) / len(scores) if scores else 0.0
    overall_status = (
        EvalStatus.PASSED if overall_score >= threshold else EvalStatus.FAILED
    )
    return EvaluationResult(
        overall_score=overall_score,
        overall_eval_status=overall_status,
        per_invocation_results=per_invocation_results,
    )

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/test_config.json
{
  "criteria": {
    "tool_trajectory_avg_score": {
      "threshold": 1.0,
      "match_type": "EXACT"
    },
    "rubric_based_tool_use_quality_v1": {
      "threshold": 0.8,
      "judge_model_options": {
        "judge_model": "gemini-3.8-flash",
        "num_samples": 3
      },
      "rubrics": [
        {
          "rubric_id": "preference_before_weather",
          "rubric_content": {
            "text_property": "When the user asks to set a temperature unit preference, the agent calls set_temperature_preference before calling get_weather."
          }
        },
        {
          "rubric_id": "all_requested_cities_queried",
          "rubric_content": {
            "text_property": "The agent calls get_weather for every city mentioned by the user."
          }
        }
      ]
    },
    "rubric_based_final_response_quality_v1": {
      "threshold": 0.8,
      "judge_model_options": {
        "judge_model": "gemini-3.8-flash",
        "num_samples": 3
      },
      "rubrics": [
        {
          "rubric_id": "accurate_temperature_and_unit",
          "rubric_content": {
            "text_property": "For supported cities, the response reports the exact temperature and unit (Celsius or Fahrenheit) returned by get_weather."
          }
        },
        {
          "rubric_id": "graceful_unsupported_city_handling",
          "rubric_content": {
            "text_property": "If get_weather returns an error for an unsupported city such as Paris, the agent politely states weather information is unavailable and does not invent a forecast."
          }
        }
      ]
    },
    "weather_unit_and_error_check": 1.0
  },
  "custom_metrics": {
    "weather_unit_and_error_check": {
      "code_config": {
        "name": "adk_agents.agent1_weather_lookup.custom_eval_metrics.weather_unit_and_error_check"
      },
      "description": "Checks °C/°F unit formatting and ensures no made-up temperatures for Paris."
    }
  }
}

In [ ]:
from adk_agents.agent1_weather_lookup import custom_eval_metrics
from google.adk.evaluation.custom_metric_evaluator import _CustomMetricEvaluator
from google.adk.evaluation.eval_metrics import Interval, MetricInfo, MetricValueInfo
from google.adk.evaluation.metric_evaluator_registry import (
    DEFAULT_METRIC_EVALUATOR_REGISTRY,
)

importlib.reload(custom_eval_metrics)

DEFAULT_METRIC_EVALUATOR_REGISTRY.register_evaluator(
    metric_info=MetricInfo(
        metric_name="weather_unit_and_error_check",
        description="Checks °C/°F unit formatting and ensures no made-up temperatures for Paris.",
        metric_value_info=MetricValueInfo(
            interval=Interval(min_value=0.0, max_value=1.0)
        ),
    ),
    evaluator=_CustomMetricEvaluator,
)

rubric_eval_df = await run_adk_eval(
    eval_dataset_file_path_or_dir="./adk_agents/agent1_weather_lookup/weather_assistant_core.test.json",
    num_runs=1,
)

display_eval_report(rubric_eval_df)

In [ ]:
display_dataframe_rows(rubric_eval_df, num_rows=4, display_drilldown=True)

In [ ]:
plot_bar_plot(
    rubric_eval_df,
    title="Custom metrics evaluation",
)

display_radar_plot(
    rubric_eval_df,
    title="Weather agent evaluation",
)

## Multi-turn evaluation

In a `.test.json` file, a test case can include multiple turns in `"conversation"` and an initial `"session_input": {"state": ...}`.

Let's create `weather_multi_turn.test.json` with two test cases:
1. **`full_stateful_weather_journey` (4 turns)**:
   * **Turn 1**: `"Hello, my name is Alex!"` $
ightarrow$ Routes to `greeting_agent` and calls `say_hello(name="Alex")`.
   * **Turn 2**: `"Please set my preferred temperature unit to Fahrenheit."` $
ightarrow$ Calls `set_temperature_preference(unit="Fahrenheit")`.
   * **Turn 3**: `"What is the weather in New York?"` $
ightarrow$ Calls `get_weather(city="New York")`, reads `"Fahrenheit"` from session state, and returns `77°F`.
   * **Turn 4**: `"Thanks for your help, goodbye!"` $
ightarrow$ Routes to `farewell_agent` and calls `say_goodbye()`.
2. **`prepopulated_fahrenheit_session`**:
   * Sets `"session_input": {"state": {"user:temperature_unit": "Fahrenheit"}}` before Turn 1 and checks that `"What is the weather like in London?"` returns `59°F`.

Using `"match_type": "IN_ORDER"` for `tool_trajectory_avg_score` checks the expected tool sequence (`say_hello` $
ightarrow$ `set_temperature_preference` $
ightarrow$ `get_weather` $
ightarrow$ `say_goodbye`) while allowing `transfer_to_agent` calls between agents.

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/weather_multi_turn.test.json
{
  "eval_set_id": "weather_assistant_multi_turn_eval_set",
  "name": "Weather Agent Multi-Turn Suite",
  "description": "Tests sub-agents (greeting_agent, farewell_agent) and session state (user:temperature_unit) across turns.",
  "eval_cases": [
    {
      "eval_id": "full_stateful_weather_journey",
      "conversation": [
        {
          "invocation_id": "turn-1-greeting",
          "user_content": {
            "parts": [{"text": "Hello, my name is Alex!"}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "Hello, Alex!"}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "say_hello", "args": {"name": "Alex"}}
            ],
            "intermediate_responses": []
          }
        },
        {
          "invocation_id": "turn-2-set-fahrenheit",
          "user_content": {
            "parts": [{"text": "Please set my preferred temperature unit to Fahrenheit."}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "Your preferred temperature unit has been set to Fahrenheit."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "set_temperature_preference", "args": {"unit": "Fahrenheit"}}
            ],
            "intermediate_responses": []
          }
        },
        {
          "invocation_id": "turn-3-ny-weather-fahrenheit",
          "user_content": {
            "parts": [{"text": "What is the weather in New York?"}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "The weather in New York is sunny with a temperature of 77°F."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "get_weather", "args": {"city": "New York"}}
            ],
            "intermediate_responses": []
          }
        },
        {
          "invocation_id": "turn-4-farewell",
          "user_content": {
            "parts": [{"text": "Thanks for your help, goodbye!"}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "Goodbye! Have a great day."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "say_goodbye", "args": {}}
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "weather_info_app",
        "user_id": "user_1",
        "state": {}
      }
    },
    {
      "eval_id": "prepopulated_fahrenheit_session",
      "conversation": [
        {
          "invocation_id": "turn-1-london-prepopulated-f",
          "user_content": {
            "parts": [{"text": "What is the weather like in London?"}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "It's cloudy in London with a temperature of 59°F."}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "get_weather", "args": {"city": "London"}}
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "weather_info_app",
        "user_id": "user_1",
        "state": {
          "user:temperature_unit": "Fahrenheit"
        }
      }
    }
  ]
}

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/test_config.json
{
  "criteria": {
    "tool_trajectory_avg_score": {
      "threshold": 1.0,
      "match_type": "IN_ORDER"
    },
    "final_response_match_v2": {
      "threshold": 0.8,
      "judge_model_options": {
        "judge_model": "gemini-3.8-flash",
        "num_samples": 3
      }
    },
    "rubric_based_multi_turn_trajectory_quality_v1": {
      "threshold": 0.8,
      "judge_model_options": {
        "judge_model": "gemini-3.8-flash",
        "num_samples": 3
      },
      "rubrics": [
        {
          "rubric_id": "stateful_unit_persistence",
          "rubric_content": {
            "text_property": "Once the user sets their temperature preference to Fahrenheit (or when Fahrenheit is set in session state), subsequent weather lookups report temperatures in Fahrenheit (°F)."
          }
        }
      ]
    }
  }
}

In [ ]:
multi_turn_eval_df = await run_adk_eval(
    eval_dataset_file_path_or_dir="./adk_agents/agent1_weather_lookup/weather_multi_turn.test.json",
    num_runs=1,
)

display_eval_report(multi_turn_eval_df)

In [ ]:
display_dataframe_rows(multi_turn_eval_df, num_rows=5, display_drilldown=True)

### Evaluate a sub-agent

You can also pass `agent_name` to `AgentEvaluator.evaluate()` to test a sub-agent (such as `greeting_agent`) directly.

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/greeting_subagent.test.json
{
  "eval_set_id": "greeting_subagent_unit_test",
  "name": "Greeting Sub-Agent Test",
  "eval_cases": [
    {
      "eval_id": "direct_greeting_alex",
      "conversation": [
        {
          "invocation_id": "inv-greet-001",
          "user_content": {
            "parts": [{"text": "Hi there, I am Alex!"}],
            "role": "user"
          },
          "final_response": {
            "parts": [{"text": "Hello, Alex!"}],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [
              {"name": "say_hello", "args": {"name": "Alex"}}
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "weather_info_app",
        "user_id": "user_1",
        "state": {}
      }
    }
  ]
}

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/test_config.json
{
  "criteria": {
    "tool_trajectory_avg_score": {
      "threshold": 1.0,
      "match_type": "EXACT"
    },
    "response_match_score": 0.5
  }
}

In [ ]:
subagent_eval_df = await run_adk_eval(
    eval_dataset_file_path_or_dir="./adk_agents/agent1_weather_lookup/greeting_subagent.test.json",
    agent_name="greeting_agent",
    num_runs=1,
)

display_eval_report(subagent_eval_df)

## Summary

In this notebook, we used `AgentEvaluator.evaluate()` to test an ADK weather agent with `.test.json` datasets and `test_config.json` criteria. We started by checking single tool calls and arguments with `tool_trajectory_avg_score`, and then compared `EXACT`, `IN_ORDER`, and `ANY_ORDER` match modes across multi-step tool sequences such as setting a temperature preference before checking the weather. When built-in metrics were not specific enough, we added LLM rubrics (`rubric_based_tool_use_quality_v1` and `rubric_based_final_response_quality_v1`) and a custom Python function (`custom_metrics`) to check domain rules like temperature units and error responses. Finally, we evaluated multi-turn conversations that carry state across turns and tested individual sub-agents directly using the `agent_name` parameter.

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.